# ML Experiment 2: Uber Fare Prediction (Linear, Ridge, Lasso, Random Forest)
**Dataset:** Kaggle *Uber Fares Dataset* (`uber.csv`). Download from https://www.kaggle.com/datasets/yasserh/uber-fares-dataset and upload it when asked below.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
import os
if not os.path.exists("uber.csv"):
    from google.colab import files
    files.upload()   # choose uber.csv
df = pd.read_csv("uber.csv")
print(df.shape)
df.head()

## Pre-processing

In [ ]:
df = df.dropna()
df = df.drop(columns=[c for c in ["Unnamed: 0", "key"] if c in df.columns])
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], errors="coerce", utc=True)
df = df.dropna()
df["hour"] = df.pickup_datetime.dt.hour
df["day"] = df.pickup_datetime.dt.day
df["month"] = df.pickup_datetime.dt.month
df["year"] = df.pickup_datetime.dt.year

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine(df.pickup_latitude, df.pickup_longitude, df.dropoff_latitude, df.dropoff_longitude)

# outlier filtering
df = df[(df.fare_amount > 0) & (df.fare_amount <= 500)]
df = df[df.pickup_longitude.between(-75, -72) & df.dropoff_longitude.between(-75, -72)]
df = df[df.pickup_latitude.between(40, 42) & df.dropoff_latitude.between(40, 42)]
df = df[df.passenger_count.between(1, 6)]
df = df[df.distance_km.between(0.1, 100)]
print("Rows after cleaning:", len(df))

## Correlation

In [ ]:
cols = ["fare_amount","passenger_count","hour","day","month","year","distance_km"]
plt.figure(figsize=(8,6))
sns.heatmap(df[cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation heatmap"); plt.show()

## Train / test split and scaling

In [ ]:
features = ["pickup_longitude","pickup_latitude","dropoff_longitude","dropoff_latitude",
            "passenger_count","hour","day","month","year","distance_km"]
X = df[features]; y = df["fare_amount"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge (alpha=1.0)": Ridge(alpha=1.0),
    "Lasso (alpha=0.1)": Lasso(alpha=0.1, max_iter=10000),
    "Random Forest": RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=42),
}
rows = []
for name, m in models.items():
    m.fit(X_train_s, y_train)
    p = m.predict(X_test_s)
    rows.append([name, mean_absolute_error(y_test,p), np.sqrt(mean_squared_error(y_test,p)), r2_score(y_test,p)])
results = pd.DataFrame(rows, columns=["Model","MAE","RMSE","R2"])
results

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4))
results.set_index("Model")[["MAE","RMSE"]].plot.bar(ax=ax[0], title="Errors (lower is better)")
results.set_index("Model")["R2"].plot.bar(ax=ax[1], color="seagreen", title="R2 (higher is better)")
for a in ax: a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## Predict a single ride (Ridge)

In [ ]:
ridge = models["Ridge (alpha=1.0)"]
sample = X_test.iloc[[0]]
print("Sample:\n", sample.T)
print("Predicted fare:", ridge.predict(scaler.transform(sample))[0])
print("Actual fare   :", y_test.iloc[0])